## USE CASE OF PANDAS DATA ANALYISIS 

You are a data engineer at the airport of noakchott and the management would like to analyse data about flights in order to answer the folowing questions 


Data acquisition

1 - Go to https://rapidapi.com/aedbx-aedbx/api/aerodatabox/playground/apiendpoint_97755564-247f-411f-bef2-ad26a453e389 <br/>
2 - Create an account(free plan)<br/>
3- Write api call to get airpot data and save it into './data/airport.json' request from (2025-01-01 untill '2025-02-02'<br/>
4- import data  <br/>


## Data analyis <br/>


Q1. How many flights does the dataset contain in total? What is the minimum data and maximum data that we have <br/>
Q2. How many distinct airlines are represented?<br/>
Q3. Fleet mix: Which aircraft models are used most and how many times?<br/>
Q4. Busiest arrival airport: List the top‑5 arrival airports by number of scheduled arrivals.<br/>
Q5. Schedule accuracy: For flights where arrival.revisedTime is available, calculate the average arrival delay (minutes).<br/>
Q6. Route mapping: Create a mini‑table with the top‑3 most common city‑pairs (origin‑>destination IATA codes) and the associated flight counts.<br/>

# GOOD LUCK

In [ ]:
print("hello Sidi Mohamed")

In [ ]:
import requests
import json
import os
from datetime import datetime, timedelta
import pytz
import time

def fetch_flights_between_dates(iata_code, start_date, end_date, output_file):
    """
    Récupère les données de vols pour un aéroport sur une période donnée
    et les enregistre dans un fichier JSON.
    
    Args:
        iata_code (str): Code IATA de l'aéroport (ex: "NKC")
        start_date (datetime): Date de début UTC
        end_date (datetime): Date de fin UTC
        output_file (str): Chemin du fichier de sortie
    """
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{iata_code}"

    headers = {
        "x-rapidapi-key": "9340651531msh24b69afeec28444p138c15jsne5d15265edcb",
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }

    now = datetime.utcnow().replace(tzinfo=pytz.utc)
    interval = timedelta(minutes=720)  # Intervalle de 12 heures
    retry_delay = 60  # Délai en secondes avant réessai après erreur 429
    max_retries = 3  # Nombre maximum de tentatives après erreur

    current_time = start_date
    all_flights = []
    quota_exceeded = False

    # Créer le répertoire de données si inexistant
    os.makedirs(os.path.dirname(output_file) or './data', exist_ok=True)

    while current_time <= end_date and not quota_exceeded:
        offset = int((current_time - now).total_seconds() / 60)
        print(f"Fetching from {current_time.isoformat()} (offset {offset} min)")

        querystring = {
            "offsetMinutes": str(offset),
            "durationMinutes": "720",
            "withLeg": "true",
            "direction": "Both",
            "withCancelled": "true",
            "withCodeshared": "true",
            "withCargo": "true",
            "withPrivate": "true",
            "withLocation": "false"
        }

        retry_count = 0
        success = False

        while retry_count < max_retries and not success and not quota_exceeded:
            try:
                response = requests.get(url, headers=headers, params=querystring)
                
                if response.status_code == 200:
                    data = response.json()
                    all_flights.extend(data.get("departures", []))
                    all_flights.extend(data.get("arrivals", []))
                    success = True
                
                elif response.status_code == 429:
                    error_data = response.json()
                    print(f"Error 429: {error_data}")
                    
                    if retry_count < max_retries - 1:
                        print(f"Quota atteint. Réessai dans {retry_delay} secondes... (Tentative {retry_count + 1}/{max_retries})")
                        time.sleep(retry_delay)
                        retry_count += 1
                    else:
                        print("Quota mensuel atteint - arrêt des requêtes")
                        quota_exceeded = True
                
                else:
                    print(f"Error {response.status_code}: {response.text}")
                    break

            except requests.exceptions.RequestException as e:
                print(f"Request failed: {e}")
                break

        current_time += interval

    try:
        with open(output_file, 'w') as f:
            json.dump(all_flights, f, indent=4)
        print(f"Flight data saved to '{output_file}'")
        print(f"Total flights fetched: {len(all_flights)}")
    except Exception as e:
        print(f"Error saving file: {e}")

In [ ]:
start_date = datetime(2025, 1, 1, tzinfo=pytz.utc)
end_date = datetime(2025, 2, 2, tzinfo=pytz.utc)
fetch_flights_between_dates("NKC", start_date, end_date, './data/airport_data_filtered.json')

#===========================================================================================

In [7]:
import json
import pandas as pd

# Function to load JSON data
def load_airpot_data():
    with open('./data/airport.json', 'r') as f: 
        data = json.load(f)
    return data

# Load data
data = load_airpot_data()

# Extract all departures
all_departures = []
for entry in data:
    if "departures" in entry:
        all_departures.extend(entry["departures"])

# Create DataFrame
df = pd.DataFrame(all_departures)

df.head()

,departure,arrival,number,status,codeshareStatus,isCargo,aircraft,airline,callSign
0,"{'scheduledTime': {'utc': '2025-01-01 21:15Z',...","{'airport': {'icao': 'GOBD', 'iata': 'DSS', 'n...",HC 206,Unknown,Unknown,False,{'model': 'ATR 72'},"{'name': 'Air Senegal', 'iata': 'HC', 'icao': ...",NaN
1,"{'scheduledTime': {'utc': '2025-01-01 23:55Z',...","{'airport': {'icao': 'LFPG', 'iata': 'CDG', 'n...",AF 769,Departed,IsOperator,False,"{'reg': 'F-HUVD', 'modeS': '39D2A3', 'model': ...","{'name': 'Air France', 'iata': 'AF', 'icao': '...",AFR769
2,"{'scheduledTime': {'utc': '2025-01-02 00:40Z',...","{'airport': {'icao': 'GCLP', 'iata': 'LPA', 'n...",NT 1805,Departed,IsOperator,False,{'model': 'E295'},"{'name': 'Binter Canarias', 'iata': 'NT', 'ica...",NaN
3,"{'scheduledTime': {'utc': '2025-01-02 02:45Z',...","{'airport': {'icao': 'GMMN', 'iata': 'CMN', 'n...",AT 510,Unknown,Unknown,False,{'model': 'Boeing 737-800'},"{'name': 'Royal Air Maroc', 'iata': 'AT', 'ica...",NaN
4,"{'scheduledTime': {'utc': '2025-01-02 07:00Z',...","{'airport': {'name': 'Nema'}, 'quality': []}",L6 20,Unknown,Unknown,False,{'model': 'Boeing 737-800'},"{'name': 'Mauritania International', 'iata': ...",NaN


Q1. How many flights does the dataset contain in total? What is the minimum data and maximum data that we have

In [8]:
# For departure (keeping your working version)
df['departure_time'] = pd.to_datetime(
    df['departure'].apply(lambda x: x['scheduledTime']['utc']), 
    errors='coerce'
)

# For arrival - with error handling
df['arrival_time'] = pd.to_datetime(
    df['arrival'].apply(lambda x: x.get('scheduledTime', {}).get('utc')), 
    errors='coerce'
)

# Calculer la date minimale et maximale pour les départs et arrivées
min_date = min(df['departure_time'].min(), df['arrival_time'].min())
max_date = max(df['departure_time'].max(), df['arrival_time'].max())

print (f"Data contains records from {min_date} to {max_date} with {len(df)} records ")

Data contains records from 2025-01-01 21:15:00+00:00 to 2025-02-19 09:30:00+00:00 with 376 records 


Q2. How many distinct airlines are represented?

In [9]:
distinct_airlines = df['airline'].apply(lambda x: x['name']).nunique()
print(f"Distinct airlines: {distinct_airlines}")

Distinct airlines: 9


Q3. Fleet mix: Which aircraft models are used most and how many times?

In [10]:
model_counts = df['aircraft'].apply(
    lambda x: x.get('model') if isinstance(x, dict) else None
).value_counts()


print(f"Most common aircraft model is {model_counts.idxmax()} with {model_counts.max()} occurrences")

Most common aircraft model is Boeing 737-800 with 68 occurrences


Q4. Busiest arrival airport: List the top‑5 arrival airports by number of scheduled arrivals.

In [11]:
top_arrivals = df['arrival'].apply(
    lambda x: x['airport']['name'] if isinstance(x, dict) else None
).value_counts().head(5)

print("Top 5 arrival airports:")
for airport, count in top_arrivals.items():
    print(f"{airport}: {count} flights")

Top 5 arrival airports:
Casablanca: 90 flights
Dakar: 64 flights
Istanbul: 34 flights
Gran Canaria Island: 34 flights
Tunis: 31 flights


Q5. Schedule accuracy: For flights where arrival.revisedTime is available, calculate the average arrival delay (minutes).

In [12]:
import pandas as pd

# Safely extract scheduled and revised times
def get_scheduled_time(x):
    try:
        return x['scheduledTime']['utc'] if isinstance(x, dict) else None
    except KeyError:
        return None

def get_revised_time(x):
    try:
        return x.get('revisedTime', {}).get('utc') if isinstance(x, dict) else None
    except KeyError:
        return None

# Convert to datetime
df['scheduled_arrival'] = pd.to_datetime(
    df['arrival'].apply(get_scheduled_time),
    errors='coerce'
)

df['revised_arrival'] = pd.to_datetime(
    df['arrival'].apply(get_revised_time),
    errors='coerce'
)

# Calculate delay only for flights with revised times
valid_delays = df['revised_arrival'].notna() & df['scheduled_arrival'].notna()
df['arrival_delay_minutes'] = (
    (df.loc[valid_delays, 'revised_arrival'] - 
     df.loc[valid_delays, 'scheduled_arrival']
    ).dt.total_seconds() / 60)

# Compute average delay
average_delay = df['arrival_delay_minutes'].mean()

print(f"Average arrival delay: {average_delay:.1f} minutes")
print(f"Based on {valid_delays.sum()} flights with revised time data")

Average arrival delay: 1.5 minutes
Based on 77 flights with revised time data


Q6. Route mapping: Create a mini‑table with the top‑3 most common city‑pairs (origin‑>destination IATA codes) and the associated flight counts.

In [13]:
# Extract origin and destination IATA codes safely
df['origin'] = df['departure'].apply(
    lambda x: x.get('airport', {}).get('iata', 'NKC') if isinstance(x, dict) else 'NKC'
)

df['destination'] = df['arrival'].apply(
    lambda x: x.get('airport', {}).get('iata') if isinstance(x, dict) else None
)

# Create route column by combining origin and destination
df['route'] = df['origin'] + '→' + df['destination']

# Count occurrences of each route and get top 3
top_routes = df['route'].value_counts().head(3)

# Convert to mini-table format
route_table = top_routes.reset_index()
route_table.columns = ['Route (Origin→Destination)', 'Flight Count']

print("Top 3 Most Common Flight Routes:")
print(route_table.to_string(index=False))

Top 3 Most Common Flight Routes:
Route (Origin→Destination)  Flight Count
                   NKC→CMN            89
                   NKC→DSS            64
                   NKC→LPA            34
